# HIV EM GESTANTES
Análise de dados de HIV em gestantes.


In [7]:
!pip install dbfread
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import warnings
from dbfread import DBF # Added for reading .dbc files

# Suppress SettingWithCopyWarning warnings, as they can be noisy
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# Note: To read .dbc files in Python, you might need a specific library (e.g., dbfread)
# or you might need to convert them to a more common format like CSV first.
# If you have a custom function to read .dbc, you can define it here:
def read_dbc_file(file_path):
    # Implement your DBC reading logic here, e.g., using an external library
    # For now, we'll assume they are converted to CSV or a similar format if not directly readable.
    return pd.DataFrame(iter(DBF(file_path))) # Using dbfread to read .dbc

In [8]:
# Definir o diretório de trabalho
# In Python, it's generally preferred to construct full file paths rather than changing the working directory globally.
# We will use `os.path.join` and `glob.glob` to find the files in 'sample_data'.

# Carregar todos os arquivos DBC do diretório
arquivos_dbc = glob.glob(os.path.join("sample_data", "*.dbc"))

lista_dados = []
for file_path in arquivos_dbc:
    try:
        # Using the read_dbc_file function defined in the previous cell
        df = read_dbc_file(file_path)
        lista_dados.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Combinar todos os arquivos em um único dataframe
if lista_dados:
    data_raw = pd.concat(lista_dados, ignore_index=True)  # Combine all DataFrames into one
    print(f"Successfully loaded and combined {len(lista_dados)} .dbc files into 'data_raw'.")
else:
    data_raw = pd.DataFrame() # Initialize an empty DataFrame if no files were read
    print("No .dbc files were loaded or combined. 'data_raw' is an empty DataFrame.")

Error reading sample_data/HIVGBR22.dbc: 'ascii' codec can't decode byte 0xd1 in position 2: ordinal not in range(128)
Error reading sample_data/HIVGBR19.dbc: 'ascii' codec can't decode byte 0xbe in position 1: ordinal not in range(128)
Successfully loaded and combined 5 .dbc files into 'data_raw'.


## Avaliar duplicatas e conversão de variáveis


In [9]:
# Verificar as colunas disponíveis
colnames(data_raw)

# Verificar duplicatas no banco de dados
duplicados <- data_raw %>%
  select(TP_NOT, ID_AGRAVO, DT_NOTIFIC, NU_ANO, SG_UF_NOT, ID_MUNICIP, ID_REGIONA, DT_DIAG, SEM_DIAG, NU_IDADE_N,
         CS_SEXO, CS_GESTANT, CS_RACA, CS_ESCOL_N, SG_UF, ID_MN_RESI, ID_RG_RESI, ID_PAIS, ANT_EVLABO, PRE_PRENAT,
         PRE_UFREL, PRE_MUNIRE, PRE_ANTRET, PRE_DT_RET, PAR_UFPART, PRE_MUNIPA, PAR_DT_PAR, PAR_TIPO, PAR_ANTIDU,
         PAR_EVOLUC, PAR_INICPR) %>%
  filter(duplicated(.))  # Identificar registros duplicados

# Exibir duplicatas encontradas
cat("Duplicatas encontradas:\n")
print(duplicados)

# Criar um novo dataframe sem duplicatas
data <- data_raw %>%
  distinct(TP_NOT, ID_AGRAVO, DT_NOTIFIC, NU_ANO, SG_UF_NOT, ID_MUNICIP, ID_REGIONA, DT_DIAG, SEM_DIAG, NU_IDADE_N,
           CS_SEXO, CS_GESTANT, CS_RACA, CS_ESCOL_N, SG_UF, ID_MN_RESI, ID_RG_RESI, ID_PAIS, ANT_EVLABO, PRE_PRENAT,
           PRE_UFREL, PRE_MUNIRE, PRE_ANTRET, PRE_DT_RET, PAR_UFPART, PRE_MUNIPA, PAR_DT_PAR, PAR_TIPO, PAR_ANTIDU,
           PAR_EVOLUC, PAR_INICPR, .keep_all = TRUE)

# Verificar a quantidade de linhas antes e depois
cat("Linhas antes: ", nrow(data_raw), "\n")
cat("Linhas depois (sem duplicatas): ", nrow(data), "\n")

#Convertendo em variaveis numericas
data$ANT_EVLABO <- as.numeric(data$ANT_EVLABO)
class(data$ANT_EVLABO)

data$PRE_PRENAT <- as.numeric(data$PRE_PRENAT)
class(data$PRE_PRENAT)

data$PAR_TIPO <- as.numeric(data$PAR_TIPO)
class(data$PAR_TIPO)

data$CS_RACA <- as.numeric(data$CS_RACA)
class(data$CS_RACA)

data$CS_ESCOL_N <- as.numeric(data$CS_ESCOL_N)
class(data$CS_ESCOL_N)

data$PAR_EVOLUC <- as.numeric(data$PAR_EVOLUC)
class(data$PAR_EVOLUC)


SyntaxError: invalid syntax (788081071.py, line 5)

## Definir o dicionário de variáveis


In [ ]:
data <- data %>%
  mutate(
    diagnostico = factor(
      ANT_EVLABO,
      levels = c("1", "2", "3", "4"),
      labels = c("Antes Pré-Natal", "Durante Pré-Natal", "Durante o parto", "Após parto")
    ),
    ano = factor(
      NU_ANO,
      levels = c(2020, 2021, 2022, 2023, 2024),
      labels = c("2020", "2021", "2022", "2023", "2024")
    ),
    preNatal = factor(
      PRE_PRENAT,
      levels = c("1", "2", "3"),
      labels = c("Sim", "Não", "Ignorado")
    ),
    tipoParto = factor(
      case_when(
        PAR_TIPO == "1" ~ "Vaginal",
        PAR_TIPO == "2" ~ "Cesárea eletiva",
        PAR_TIPO == "3" ~ "Cesárea Urgência",
        PAR_TIPO == "4" ~ NA_character_,  # Ignorado transformado em NA
        TRUE ~ NA_character_  # Outros valores fora do esperado também como NA
      ),
      levels = c("Vaginal", "Cesárea eletiva", "Cesárea Urgência")
    ),
    cs_raca = factor(
      case_when(
        CS_RACA == 1 ~ "Branca",
        CS_RACA == 2 ~ "Preta",
        CS_RACA == 3 ~ "Amarela",
        CS_RACA == 4 ~ "Parda",
        CS_RACA == 5 ~ "Indígena",
        TRUE ~ NA_character_  # Ignorado e outros valores viram NA
      ),
      levels = c("Branca", "Preta", "Amarela", "Parda", "Indígena")
    ),
    cs_escol_n = factor(
      case_when(
        CS_ESCOL_N == 43 ~ "Analfabeto",
        CS_ESCOL_N %in% c(1, 2, 3) ~ "Fundamental Incompleto",
        CS_ESCOL_N == 4 ~ "Fundamental Completo", #4,5
        CS_ESCOL_N == 5 ~ "Médio Incompleto",
        CS_ESCOL_N == 6 ~ "Médio Completo", #6,7
        CS_ESCOL_N == 7 ~ "Superior Incompleto",
        CS_ESCOL_N == 8 ~ "Superior Completo", # 8
        CS_ESCOL_N %in% c(9, 10) ~ NA_character_,  # Ignorado e NA transformados em NA
        TRUE ~ NA_character_  # Outros valores fora do esperado também viram NA
      ),
      levels = c(
        "Analfabeto",
        "Fundamental Incompleto",
        "Fundamental Completo",
        "Médio Incompleto",
        "Médio Completo",
        "Superior Incompleto",
        "Superior Completo"
      )
    ),
    # Corrigir e categorizar idade
    NU_IDADE_N = case_when(
      NU_IDADE_N <= 4000 ~ 0,  # Supondo que valores <= 4000 sejam categorias específicas
      NU_IDADE_N > 4000 ~ as.numeric(substring(as.character(NU_IDADE_N), 2, 4)),  # Extrair idade
      TRUE ~ NA_real_
    ),
    idade_categoria = case_when(
      NU_IDADE_N >= 10 & NU_IDADE_N <= 19 ~ "Adolescente (10 - 19 anos)",
      NU_IDADE_N >= 20 & NU_IDADE_N <= 29 ~ "Jovem Adulta (20 - 29 anos)",
      NU_IDADE_N >= 30 & NU_IDADE_N <= 39 ~ "Adulta (30 - 39 anos)",
      NU_IDADE_N >= 40 ~ "Adulta (acima de 40)",
      TRUE ~ "N/A"  # Caso haja valores ausentes ou fora do intervalo esperado
    ),
    idade_categoria = factor(
      idade_categoria,
      levels = c(
        "Adolescente (10 - 19 anos)",
        "Jovem Adulta (20 - 29 anos)",
        "Adulta (30 - 39 anos)",
        "Adulta (acima de 40)",
        "N/A"
      )
    )
  )

# Verificar a distribuição das variáveis
data %>% count(diagnostico)
data %>% count(ano)
data %>% count(preNatal)
data %>% count(tipoParto)
data %>% count(cs_raca)
data %>% count(cs_escol_n)
data %>% count(idade_categoria)


## Tabela Escolaridade, raça, ano, idade


In [ ]:
data %>%
  select(idade_categoria,cs_raca,cs_escol_n,ano) %>%
  filter()%>%
  droplevels()%>%
  tbl_summary(
    by = ano
  ) %>%
  modify_header(label = "**Variável**") %>%
  modify_table_body( #modifica as linhas dos indices
    ~ .x %>%
      mutate(label = case_when(
        label == "cs_raca" ~ "Raça/Cor",
        label == "cs_escol_n" ~ "Escolaridade",
        label == "idade_categoria" ~ "Idade",
        label == "Unknown" ~ "N/A",
        TRUE ~ label  # Mantém os outros rótulos
      ))
  )%>%
  bold_labels() %>%
  as_flex_table() #tem que desativar para salvar
#as_gt() %>% ## Salvando a Tabela
#gt::gtsave(filename = "tabela1.png") # use extensions .png, .html, .docx, .rtf, .tex, .ltx


## Casos acumulados por região/ano


In [ ]:
# Adicionar a classificação por região
data <- data %>%
  mutate(
    regiao = case_when(
      SG_UF %in% c(12, 13, 16, 15, 11, 14, 17) ~ "Norte",
      SG_UF %in% c(27, 29, 23, 21, 25, 26, 22, 24, 28) ~ "Nordeste",
      SG_UF %in% c(52, 51, 50) ~ "Centro-Oeste",
      SG_UF %in% c(53) ~ "Distrito Federal",
      SG_UF %in% c(32, 31, 33, 35) ~ "Sudeste",
      SG_UF %in% c(41, 42, 43) ~ "Sul",
      TRUE ~ "N/A"
    ),
    regiao = factor(regiao,
                    levels = c("Centro-Oeste",
                               "Distrito Federal",
                               "Nordeste",
                               "Norte",
                               "Sudeste",
                               "Sul",
                               "N/A")
     ))

# Agregar os dados por ano e região
dados_agrupados_regiao <- data %>%
  filter(regiao != "N/A") %>%  # Filtrar para excluir "N/A"
  group_by(ano, regiao) %>%
  summarise(casos_totais = n(), .groups = "drop")

# Gráfico de casos por região e ano
ggplot(dados_agrupados_regiao, aes(x = ano, y = casos_totais, color = regiao, group = regiao)) +
  geom_line(size = 1) +
  geom_point(size = 3) +
  labs(
    title = "Casos Totais de HIV em Gestantes por Região e Ano",
    x = "Ano",
    y = "Casos Totais",
    color = "Região"
  ) +
  theme_minimal()


## Q-square idade x pre-natal


In [ ]:
# Remover linhas com NA antes de criar a tabela de contingência
data_filtrada <- data %>%
  filter(!is.na(idade_categoria) & !is.na(preNatal))

# Remover níveis não utilizados após filtrar os dados
data_filtrada$idade_categoria <- droplevels(data_filtrada$idade_categoria)
data_filtrada$preNatal <- droplevels(data_filtrada$preNatal)

# Criar a tabela de contingência com dados filtrados
tabela_contingencia <- table(data_filtrada$idade_categoria, data_filtrada$preNatal)

# Filtrar a tabela para remover "Ignorado" e "N/A"
tabela_contingencia_filtrada <- tabela_contingencia[!rownames(tabela_contingencia) %in% c("N/A", "Ignorado"), c("Sim", "Não")]

# Exibir a tabela de contingência
cat("Tabela de Contingência (sem Ignorado e N/A):\n")
print(tabela_contingencia_filtrada)

# Realizar o teste do qui-quadrado
teste_qui_quadrado <- chisq.test(tabela_contingencia_filtrada)

# Exibir os resultados do teste
cat("\nResultado do Teste do Qui-Quadrado:\n")
print(teste_qui_quadrado)

# Verificar as frequências esperadas
cat("\nFrequências Esperadas:\n")
print(teste_qui_quadrado$expected)

# Identificar quais células têm frequências esperadas menores que 5
freqs_esperadas <- teste_qui_quadrado$expected
cat("\nFrequências esperadas menores que 5:\n")
print(freqs_esperadas[freqs_esperadas < 5])

#Tabela Qui-square
tabela_df <- as.data.frame(tabela_contingencia_filtrada)

# Calcular o valor de p do teste do qui-quadrado
p_value <- round(teste_qui_quadrado$p.value, 4)

# Adicionar uma coluna para as variáveis (Sim/Não)
colnames(tabela_df) <- c("Idade_Categoria", "Resposta", "Frequencia")

# Criar o gráfico de barras
grafico <- ggplot(tabela_df, aes(x = Idade_Categoria, y = Frequencia, fill = Resposta)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Distribuição de Respostas por Idade",
       x = "Categoria de Idade",
       y = "Frequência",
       fill = "Resposta") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  annotate("text", x = 3, y = max(tabela_df$Frequencia) * 0.9,
           label = paste("p-value =", p_value), size = 5, color = "black")+
  theme_minimal()

# Exibir o gráfico
print(grafico)


## Q-Quadrado Tipo de parto X preNatal


In [ ]:
# Criar tabela de contingência
tabela_contingencia <- data %>%
  filter(tipoParto != "Ignorado" & preNatal != "Ignorado") %>%
  count(preNatal, tipoParto) %>%
  pivot_wider(names_from = tipoParto, values_from = n, values_fill = 0) %>%
  column_to_rownames("preNatal")

# Realizar o teste qui-quadrado
teste_qui2 <- chisq.test(tabela_contingencia)

# Exibir os resultados do teste
teste_qui2

data %>%
  filter(tipoParto != "Ignorado" & preNatal != "Ignorado") %>%
  count(preNatal, tipoParto) %>%
  group_by(preNatal) %>%
  mutate(percentual = n / sum(n) * 100) %>%
  ggplot(aes(x = preNatal, y = percentual, fill = tipoParto)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(
    title = "Relação entre Tipo de Parto e Pré-Natal (p-value < 2.2e-16)",
    x = "Pré-Natal",
    y = "Percentual (%)",
    fill = "Tipo de Parto"
  ) +
  theme_minimal()


## Momento do Diagnóstico x idade


In [ ]:
# Filtrar dados para remover valores "Ignorado" nas variáveis relevantes
data_filtrada <- data %>%
  filter(!is.na(diagnostico) & !is.na(idade_categoria))

# Criar tabela de contingência entre diagnóstico (ANT_EVLABO) e idade_categoria
tabela_contingencia <- table(data_filtrada$diagnostico, data_filtrada$idade_categoria)

# Criar o gráfico de barras
grafico <- ggplot(as.data.frame(tabela_contingencia), aes(x = Var1, y = Freq, fill = Var2)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Distribuição de Diagnóstico por Idade Categoria",
       x = "Diagnóstico de HIV",
       y = "Frequência",
       fill = "Idade Categoria") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme_minimal()

# Exibir o gráfico
print(grafico)


## Avaliar idade X pre natal (Q-square)


In [ ]:
# Remover linhas com NA antes de criar a tabela de contingência
data_filtrada <- data %>%
  filter(!is.na(cs_escol_n) & !is.na(preNatal))

# Remover níveis não utilizados após filtrar os dados
data_filtrada$cs_escol_n <- droplevels(data_filtrada$cs_escol_n)
data_filtrada$preNatal <- droplevels(data_filtrada$preNatal)

# Criar a tabela de contingência com dados filtrados
tabela_contingencia <- table(data_filtrada$cs_escol_n, data_filtrada$preNatal)

# Filtrar a tabela para remover "Ignorado" e "N/A"
tabela_contingencia_filtrada <- tabela_contingencia[!rownames(tabela_contingencia) %in% c("N/A", "Ignorado"), c("Sim", "Não")]

# Exibir a tabela de contingência
cat("Tabela de Contingência (sem Ignorado e N/A):\n")
print(tabela_contingencia_filtrada)

# Realizar o teste do qui-quadrado
teste_qui_quadrado <- chisq.test(tabela_contingencia_filtrada)

# Exibir os resultados do teste
cat("\nResultado do Teste do Qui-Quadrado:\n")
print(teste_qui_quadrado)

# Verificar as frequências esperadas
cat("\nFrequências Esperadas:\n")
print(teste_qui_quadrado$expected)

# Identificar quais células têm frequências esperadas menores que 5
freqs_esperadas <- teste_qui_quadrado$expected
cat("\nFrequências esperadas menores que 5:\n")
print(freqs_esperadas[freqs_esperadas < 5])

#Tabela Qui-square
tabela_df2 <- as.data.frame(tabela_contingencia_filtrada)

# Calcular o valor de p do teste do qui-quadrado
p_value <- round(teste_qui_quadrado$p.value, 4)


# Adicionar uma coluna para as variáveis (Sim/Não)
colnames(tabela_df2) <- c("Idade_Categoria", "Resposta", "Frequencia")

# Criar o gráfico de barras
grafico <- ggplot(tabela_df2, aes(x = Idade_Categoria, y = Frequencia, fill = Resposta)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Distribuição de Respostas por Idade",
       x = "Categoria de Idade",
       y = "Frequência",
       fill = "Resposta") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme_minimal()

# Exibir o gráfico
print(grafico)
